# Basic Time Series Visualization

This notebook covers fundamental techniques for visualizing time series data. We'll explore various approaches to effectively display and analyze temporal data patterns, trends, and seasonality using different Python visualization libraries.

## 1. Import Required Libraries

First, let's import the essential libraries for time series visualization:

In [ ]:
# Import core libraries for data handling and visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from statsmodels.tsa.seasonal import seasonal_decompose
import warnings

# Set plotting styles and configurations
plt.style.use('seaborn-v0_8-whitegrid')
warnings.filterwarnings('ignore')

# Set figure size for better readability
plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['figure.dpi'] = 100

# Show plots inline in the notebook
%matplotlib inline

## 2. Working with Time Series Data

Let's create some sample time series data to work with. We'll generate:

1. Daily stock price data
2. Monthly temperature data
3. Hourly website traffic data

In [ ]:
# Generate sample time series data

# Sample 1: Stock price data (daily)
date_range_daily = pd.date_range(start='2022-01-01', end='2022-12-31', freq='D')
np.random.seed(42)  # For reproducibility

# Create a starting price and random walk
base_price = 100
random_walk = np.random.normal(0, 1, len(date_range_daily))
random_walk = random_walk.cumsum()  # Cumulative sum for random walk

# Add seasonality and trend
trend = np.linspace(0, 20, len(date_range_daily))  # Upward trend
seasonality = 5 * np.sin(np.linspace(0, 12*np.pi, len(date_range_daily)))  # Seasonal pattern

stock_prices = base_price + random_walk + trend + seasonality
stock_df = pd.DataFrame({
    'date': date_range_daily,
    'price': stock_prices
})
stock_df.set_index('date', inplace=True)

# Sample 2: Monthly temperature data
date_range_monthly = pd.date_range(start='2018-01-01', end='2022-12-31', freq='M')
base_temp = 15
seasonality = 10 * np.sin(np.linspace(0, 10*np.pi, len(date_range_monthly)))  # Stronger seasonal pattern
noise = np.random.normal(0, 1, len(date_range_monthly))
trend = np.linspace(0, 2, len(date_range_monthly))  # Slight warming trend

temp_data = base_temp + seasonality + noise + trend
temp_df = pd.DataFrame({
    'date': date_range_monthly,
    'temperature': temp_data
})
temp_df.set_index('date', inplace=True)

# Sample 3: Hourly website traffic data (for a week)
date_range_hourly = pd.date_range(start='2023-01-01', periods=24*7, freq='H')
base_traffic = 100
 
# Daily pattern with two peaks (morning and evening)
hour_of_day = np.array([i.hour for i in date_range_hourly])
daily_pattern = 50 * np.sin(np.pi * hour_of_day / 12)**2
 
# Weekly pattern (weekends have higher traffic)
day_of_week = np.array([i.dayofweek for i in date_range_hourly])
weekend_boost = 30 * np.isin(day_of_week, [5, 6]).astype(int)
 
# Random noise
noise = np.random.normal(0, 10, len(date_range_hourly))
 
traffic_data = base_traffic + daily_pattern + weekend_boost + noise
traffic_df = pd.DataFrame({
    'datetime': date_range_hourly,
    'visits': traffic_data
})
traffic_df.set_index('datetime', inplace=True)

# Display the first few rows of each dataset
print("Stock Price Data (Daily):")
print(stock_df.head())
print("\nTemperature Data (Monthly):")
print(temp_df.head())
print("\nWebsite Traffic Data (Hourly):")
print(traffic_df.head())

### 2.1 Handling Missing Values in Time Series Data

Time series data often contains missing values. Let's create some gaps in our data and demonstrate how to handle them:

In [ ]:
# Create a copy of the stock price data with some missing values
stock_with_gaps = stock_df.copy()

# Randomly remove some values (create gaps)
np.random.seed(42)
drop_indices = np.random.choice(stock_with_gaps.index, size=30, replace=False)
stock_with_gaps.loc[drop_indices, 'price'] = np.nan

# Display the data with gaps
print("Data with missing values:")
print(stock_with_gaps.isna().sum())
print(stock_with_gaps.head(10))

# Different methods for handling missing values
# 1. Forward fill
stock_ffill = stock_with_gaps.ffill()

# 2. Backward fill
stock_bfill = stock_with_gaps.bfill()

# 3. Linear interpolation
stock_interp = stock_with_gaps.interpolate(method='linear')

# Visualize the different filling methods
plt.figure(figsize=(14, 8))

# Plot the original data with gaps
plt.subplot(2, 2, 1)
stock_with_gaps['price'].plot(color='gray', marker='o', linestyle='-', markersize=4)
plt.title('Original Data with Gaps')
plt.ylabel('Stock Price')

# Plot forward fill
plt.subplot(2, 2, 2)
stock_ffill['price'].plot(color='green', marker='o', linestyle='-', markersize=4)
plt.title('Forward Fill (ffill)')
plt.ylabel('Stock Price')

# Plot backward fill
plt.subplot(2, 2, 3)
stock_bfill['price'].plot(color='red', marker='o', linestyle='-', markersize=4)
plt.title('Backward Fill (bfill)')
plt.ylabel('Stock Price')

# Plot interpolation
plt.subplot(2, 2, 4)
stock_interp['price'].plot(color='blue', marker='o', linestyle='-', markersize=4)
plt.title('Linear Interpolation')
plt.ylabel('Stock Price')

plt.tight_layout()
plt.show()

# We'll continue with the interpolated data
stock_df_clean = stock_interp

### 2.2 Resampling Time Series Data

Resampling is a common technique for changing the frequency of time series data. Let's demonstrate upsampling (to higher frequency) and downsampling (to lower frequency):

In [ ]:
# Example of downsampling (from daily to weekly, monthly, quarterly)
stock_weekly = stock_df.resample('W').mean()  # Weekly average
stock_monthly = stock_df.resample('M').mean()  # Monthly average
stock_quarterly = stock_df.resample('Q').mean()  # Quarterly average

# Plot the different frequencies
plt.figure(figsize=(14, 10))

plt.subplot(4, 1, 1)
stock_df['price'].plot(color='blue')
plt.title('Original Daily Data')
plt.ylabel('Price')

plt.subplot(4, 1, 2)
stock_weekly['price'].plot(color='green', marker='o')
plt.title('Weekly Resampled Data')
plt.ylabel('Price')

plt.subplot(4, 1, 3)
stock_monthly['price'].plot(color='red', marker='o')
plt.title('Monthly Resampled Data')
plt.ylabel('Price')

plt.subplot(4, 1, 4)
stock_quarterly['price'].plot(color='purple', marker='o')
plt.title('Quarterly Resampled Data')
plt.ylabel('Price')

plt.tight_layout()
plt.show()

# Example of upsampling (from monthly to daily)
# First, let's resample the temperature data from monthly to daily
temp_daily = temp_df.resample('D').interpolate(method='cubic')  # Interpolate between monthly points

# Plot the original and upsampled data
plt.figure(figsize=(12, 6))

plt.plot(temp_df.index, temp_df['temperature'], 'o-', color='blue', label='Original Monthly Data')
plt.plot(temp_daily.index, temp_daily['temperature'], '-', color='red', alpha=0.7, label='Daily Interpolation')
plt.title('Upsampling: Monthly to Daily Temperature Data')
plt.legend()
plt.ylabel('Temperature (°C)')
plt.grid(True, alpha=0.3)

plt.show()

## 3. Line Plots for Time Series

Basic line plots are the most common visualization for time series data. Let's explore different ways to create them:

In [ ]:
# Basic time series line plot using Matplotlib
plt.figure(figsize=(12, 6))
plt.plot(stock_df.index, stock_df['price'], color='blue', linewidth=1.5)
plt.title('Stock Price over Time (2022)', fontsize=14)
plt.ylabel('Price ($)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Using Pandas built-in plotting (based on Matplotlib)
stock_df['price'].plot(figsize=(12, 6), color='darkblue', linewidth=1.5)
plt.title('Stock Price over Time using Pandas Plot', fontsize=14)
plt.ylabel('Price ($)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Formatting the x-axis for better readability with different date frequencies
fig, axes = plt.subplots(3, 1, figsize=(12, 12))

# Daily data with monthly ticks
stock_df['price'].plot(ax=axes[0], color='green')
axes[0].set_title('Daily Stock Price with Monthly Ticks')
axes[0].set_ylabel('Price ($)')
# Format x-axis to show months
axes[0].xaxis.set_major_locator(plt.matplotlib.dates.MonthLocator())
axes[0].xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b %Y'))

# Monthly data
temp_df['temperature'].plot(ax=axes[1], color='red', marker='o')
axes[1].set_title('Monthly Temperature Data')
axes[1].set_ylabel('Temperature (°C)')
# Format x-axis to show years and months
axes[1].xaxis.set_major_locator(plt.matplotlib.dates.YearLocator())
axes[1].xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%Y'))
axes[1].xaxis.set_minor_locator(plt.matplotlib.dates.MonthLocator())
axes[1].grid(True, which='minor', alpha=0.2)

# Hourly data
traffic_df['visits'].iloc[:24*3].plot(ax=axes[2], color='purple')
axes[2].set_title('Hourly Website Traffic (First 3 Days)')
axes[2].set_ylabel('Visitors')
# Format x-axis to show days and hours
axes[2].xaxis.set_major_locator(plt.matplotlib.dates.DayLocator())
axes[2].xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%a %d'))
axes[2].xaxis.set_minor_locator(plt.matplotlib.dates.HourLocator(interval=6))
axes[2].xaxis.set_minor_formatter(plt.matplotlib.dates.DateFormatter('%H:%M'))
axes[2].grid(True, which='minor', alpha=0.2)

plt.tight_layout()
plt.show()

## 4. Customizing Time Series Visualizations

Let's enhance our time series plots with additional features like trend lines, moving averages, and annotations:

In [ ]:
# Calculate moving averages
stock_df['MA7'] = stock_df['price'].rolling(window=7).mean()  # 7-day moving average
stock_df['MA30'] = stock_df['price'].rolling(window=30).mean()  # 30-day moving average

# Create the plot with the original data and moving averages
plt.figure(figsize=(14, 7))
plt.plot(stock_df.index, stock_df['price'], label='Daily Price', color='gray', alpha=0.5, linewidth=1)
plt.plot(stock_df.index, stock_df['MA7'], label='7-Day MA', color='blue', linewidth=2)
plt.plot(stock_df.index, stock_df['MA30'], label='30-Day MA', color='red', linewidth=2)

# Add annotations for key events
events = {
    '2022-03-15': 'Market Drop',
    '2022-06-01': 'Recovery',
    '2022-09-20': 'Peak',
    '2022-11-15': 'Decline'
}

for date, label in events.items():
    price = stock_df.loc[date, 'price']
    plt.annotate(label, xy=(pd.to_datetime(date), price), 
                xytext=(15, 15), textcoords='offset points',
                arrowprops=dict(arrowstyle='->', color='black', lw=1),
                fontsize=10, color='darkblue')

plt.title('Stock Price with Moving Averages and Key Events', fontsize=14)
plt.ylabel('Price ($)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

# Example with dual y-axes for comparing series with different scales
# Let's create a hypothetical trading volume series
np.random.seed(42)
volume_base = np.random.randint(1000, 5000, size=len(stock_df))
seasonal_factor = 1 + 0.5 * np.sin(np.linspace(0, 4*np.pi, len(stock_df)))
stock_df['volume'] = (volume_base * seasonal_factor).astype(int)

# Plot price and volume on dual axes
fig, ax1 = plt.subplots(figsize=(14, 7))

# First y-axis (left) - Price
color = 'tab:blue'
ax1.set_xlabel('Date')
ax1.set_ylabel('Price ($)', color=color)
ax1.plot(stock_df.index, stock_df['price'], color=color)
ax1.tick_params(axis='y', labelcolor=color)

# Second y-axis (right) - Volume
ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Volume', color=color)
ax2.bar(stock_df.index, stock_df['volume'], alpha=0.2, color=color)
ax2.tick_params(axis='y', labelcolor=color)

# Add a title
fig.tight_layout()
plt.title('Stock Price and Trading Volume', fontsize=14, pad=20)
plt.show()

## 5. Handling Date Ranges and Frequencies

In this section, we'll explore how to visualize specific date ranges and work with different time frequencies:

In [ ]:
# Visualizing specific date ranges
Q1_2022 = stock_df['2022-01-01':'2022-03-31']
Q2_2022 = stock_df['2022-04-01':'2022-06-30']
H2_2022 = stock_df['2022-07-01':'2022-12-31']

plt.figure(figsize=(14, 10))

# Full year
plt.subplot(4, 1, 1)
plt.plot(stock_df.index, stock_df['price'], color='black')
plt.title('Full Year 2022')
plt.ylabel('Price ($)')
plt.grid(True, alpha=0.3)

# Q1
plt.subplot(4, 1, 2)
plt.plot(Q1_2022.index, Q1_2022['price'], color='blue')
plt.title('Q1 2022')
plt.ylabel('Price ($)')
plt.grid(True, alpha=0.3)

# Q2
plt.subplot(4, 1, 3)
plt.plot(Q2_2022.index, Q2_2022['price'], color='green')
plt.title('Q2 2022')
plt.ylabel('Price ($)')
plt.grid(True, alpha=0.3)

# H2
plt.subplot(4, 1, 4)
plt.plot(H2_2022.index, H2_2022['price'], color='red')
plt.title('H2 2022')
plt.ylabel('Price ($)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Working with different time frequencies
# Weekly pattern visualization for hourly data

# Calculate the average traffic by hour of day
hourly_avg = traffic_df.groupby(traffic_df.index.hour)['visits'].mean()

# Calculate the average traffic by day of week
daily_avg = traffic_df.groupby(traffic_df.index.dayofweek)['visits'].mean()
day_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

plt.figure(figsize=(14, 6))

# Hourly pattern
plt.subplot(1, 2, 1)
plt.bar(hourly_avg.index, hourly_avg.values, color='skyblue')
plt.title('Average Traffic by Hour of Day')
plt.xlabel('Hour')
plt.ylabel('Average Visitors')
plt.xticks(range(0, 24, 3))
plt.grid(True, axis='y', alpha=0.3)

# Daily pattern
plt.subplot(1, 2, 2)
plt.bar(range(7), daily_avg.values, color='lightgreen')
plt.title('Average Traffic by Day of Week')
plt.xlabel('Day')
plt.ylabel('Average Visitors')
plt.xticks(range(7), day_names, rotation=45)
plt.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Visualizing Multiple Time Series

Now let's explore techniques for comparing multiple time series:

In [ ]:
# Create additional stock series for comparison
np.random.seed(42)
stock_df['stock_B'] = stock_df['price'] * 0.7 + np.random.normal(0, 5, len(stock_df))
stock_df['stock_C'] = stock_df['price'] * 1.3 + np.random.normal(0, 8, len(stock_df)) - 30

# Create a DataFrame with the three stocks
stocks_comparison = stock_df[['price', 'stock_B', 'stock_C']].rename(
    columns={'price': 'stock_A'}
)

# Plot all three stocks on the same axes
plt.figure(figsize=(12, 6))
for col, color in zip(stocks_comparison.columns, ['blue', 'green', 'red']):
    plt.plot(stocks_comparison.index, stocks_comparison[col], label=col, color=color, alpha=0.7)

plt.title('Comparison of Three Stocks', fontsize=14)
plt.ylabel('Price ($)', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Using subplots for clearer comparison
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

for i, (col, color) in enumerate(zip(stocks_comparison.columns, ['blue', 'green', 'red'])):
    axes[i].plot(stocks_comparison.index, stocks_comparison[col], color=color)
    axes[i].set_title(f'{col} Price', fontsize=12)
    axes[i].set_ylabel('Price ($)')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Area chart for stacked time series (useful for components that add up)
# Create data representing market segments
market_data = pd.DataFrame({
    'date': pd.date_range(start='2022-01-01', end='2022-12-31', freq='D'),
    'segment_A': np.random.randint(300, 500, 365) + 30 * np.sin(np.linspace(0, 2*np.pi, 365)),
    'segment_B': np.random.randint(200, 300, 365) + 20 * np.sin(np.linspace(0, 4*np.pi, 365)),
    'segment_C': np.random.randint(100, 200, 365) + 10 * np.sin(np.linspace(0, 6*np.pi, 365))
})
market_data.set_index('date', inplace=True)

# Simple stacked area chart
plt.figure(figsize=(14, 7))
plt.stackplot(market_data.index, 
              market_data['segment_A'], 
              market_data['segment_B'], 
              market_data['segment_C'],
              labels=['Segment A', 'Segment B', 'Segment C'],
              colors=['#FFA07A', '#20B2AA', '#9370DB'],
              alpha=0.8)
plt.title('Market Segments over Time', fontsize=14)
plt.ylabel('Market Size', fontsize=12)
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Percentage stacked area chart
market_pct = market_data.divide(market_data.sum(axis=1), axis=0) * 100

plt.figure(figsize=(14, 7))
plt.stackplot(market_pct.index, 
              market_pct['segment_A'], 
              market_pct['segment_B'], 
              market_pct['segment_C'],
              labels=['Segment A', 'Segment B', 'Segment C'],
              colors=['#FFA07A', '#20B2AA', '#9370DB'],
              alpha=0.8)
plt.title('Market Segments Share (%) over Time', fontsize=14)
plt.ylabel('Market Share (%)', fontsize=12)
plt.ylim(0, 100)
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Seasonal Decomposition Plots

Decomposing a time series into its trend, seasonal, and residual components can provide valuable insights:

In [ ]:
# Seasonal decomposition of the stock data
# Resample to weekly data for clearer seasonal patterns
weekly_data = stock_df['price'].resample('W').mean()

# Perform seasonal decomposition
decomposition = seasonal_decompose(weekly_data, model='additive', period=52)

# Plot the decomposition
fig, (ax1, ax2, ax3, ax4) = plt.subplots(4, 1, figsize=(14, 10))

# Original data
decomposition.observed.plot(ax=ax1)
ax1.set_title('Original Time Series')
ax1.set_ylabel('Price ($)')

# Trend component
decomposition.trend.plot(ax=ax2)
ax2.set_title('Trend Component')
ax2.set_ylabel('Trend')

# Seasonal component
decomposition.seasonal.plot(ax=ax3)
ax3.set_title('Seasonal Component')
ax3.set_ylabel('Seasonality')

# Residual component
decomposition.resid.plot(ax=ax4)
ax4.set_title('Residual Component')
ax4.set_ylabel('Residuals')

plt.tight_layout()
plt.show()

# Now, let's perform seasonal decomposition on the temperature data
# Monthly temperature data already has a strong seasonal pattern
temp_decomposition = seasonal_decompose(temp_df['temperature'], model='additive', period=12)

# Plot the decomposition with a more attractive style
plt.style.use('seaborn-v0_8-whitegrid')
fig = plt.figure(figsize=(14, 10))

# Original data
ax1 = plt.subplot(411)
ax1.plot(temp_decomposition.observed.index, temp_decomposition.observed.values, color='#1E88E5')
ax1.set_ylabel('Temperature (°C)', fontsize=10)
ax1.set_title('Original Temperature Time Series', fontsize=12)
ax1.fill_between(temp_decomposition.observed.index, temp_decomposition.observed.values, 
                 alpha=0.2, color='#1E88E5')

# Trend component
ax2 = plt.subplot(412)
ax2.plot(temp_decomposition.trend.index, temp_decomposition.trend.values, color='#D81B60')
ax2.set_ylabel('Trend', fontsize=10)
ax2.set_title('Trend Component', fontsize=12)
ax2.fill_between(temp_decomposition.trend.index, temp_decomposition.trend.values, 
                 alpha=0.2, color='#D81B60')

# Seasonal component
ax3 = plt.subplot(413)
ax3.plot(temp_decomposition.seasonal.index, temp_decomposition.seasonal.values, color='#8E24AA')
ax3.set_ylabel('Seasonality', fontsize=10)
ax3.set_title('Seasonal Component', fontsize=12)
ax3.fill_between(temp_decomposition.seasonal.index, temp_decomposition.seasonal.values, 
                 alpha=0.2, color='#8E24AA')

# Residual component
ax4 = plt.subplot(414)
ax4.plot(temp_decomposition.resid.index, temp_decomposition.resid.values, color='#FFC107')
ax4.set_ylabel('Residuals', fontsize=10)
ax4.set_title('Residual Component', fontsize=12)
ax4.fill_between(temp_decomposition.resid.index, temp_decomposition.resid.values, 
                 alpha=0.2, color='#FFC107')

plt.tight_layout()
plt.show()

## 8. Creating Interactive Time Series Visualizations

Interactive visualizations can provide deeper insights by allowing users to explore the data:

In [ ]:
# Create an interactive time series plot with Plotly Express
fig = px.line(stock_df.reset_index(), x='date', y='price', 
             title='Interactive Stock Price Visualization (2022)',
             labels={'date': 'Date', 'price': 'Price ($)'},
             line_shape='linear')

fig.update_layout(
    xaxis_rangeslider_visible=True,  # Add a range slider
    width=900,
    height=500
)

fig.show()

# Create a more complex interactive plot with multiple series
fig = px.line(stocks_comparison.reset_index(), x='date', 
              y=['stock_A', 'stock_B', 'stock_C'],
              title='Interactive Comparison of Three Stocks',
              labels={'date': 'Date', 'value': 'Price ($)', 'variable': 'Stock'},
              line_shape='linear')

fig.update_layout(
    xaxis_rangeslider_visible=True,  # Add a range slider
    width=900,
    height=500,
    hovermode="x unified"  # Show all values at the same x position
)

fig.show()

# Create an interactive area chart for market segments
fig = px.area(market_data.reset_index(), x='date', 
              y=['segment_A', 'segment_B', 'segment_C'],
              title='Interactive Market Segments Visualization',
              labels={'date': 'Date', 'value': 'Market Size', 'variable': 'Segment'},
              color_discrete_map={
                  'segment_A': '#FFA07A',
                  'segment_B': '#20B2AA', 
                  'segment_C': '#9370DB'
              })

fig.update_layout(
    xaxis_rangeslider_visible=True,
    width=900,
    height=500,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.show()

# Create a more advanced interactive visualization with annotations and buttons
# We'll use plotly graph objects for more customization

# First, let's add some event dates we want to highlight
events = {
    '2022-03-15': {'name': 'Market Drop', 'description': 'Significant price decrease due to market concerns'},
    '2022-06-01': {'name': 'Recovery', 'description': 'Market begins recovery after prolonged downturn'},
    '2022-09-20': {'name': 'Peak', 'description': 'Stock reaches highest point of the year'},
    '2022-11-15': {'name': 'Decline', 'description': 'Start of year-end decline phase'}
}

# Create the figure
fig = go.Figure()

# Add the main stock price line
fig.add_trace(
    go.Scatter(
        x=stock_df.index, 
        y=stock_df['price'],
        mode='lines',
        name='Stock A',
        line=dict(color='royalblue', width=2)
    )
)

# Add moving averages
fig.add_trace(
    go.Scatter(
        x=stock_df.index, 
        y=stock_df['MA7'],
        mode='lines',
        name='7-Day MA',
        line=dict(color='firebrick', width=1.5, dash='dot'),
        visible='legendonly'  # Hidden by default, can be toggled on
    )
)

fig.add_trace(
    go.Scatter(
        x=stock_df.index, 
        y=stock_df['MA30'],
        mode='lines',
        name='30-Day MA',
        line=dict(color='green', width=2, dash='dash'),
        visible='legendonly'  # Hidden by default, can be toggled on
    )
)

# Add annotations for key events
annotations = []
for date, event_info in events.items():
    annotations.append(
        dict(
            x=date,
            y=stock_df.loc[date, 'price'],
            xref="x",
            yref="y",
            text=event_info['name'],
            showarrow=True,
            arrowhead=2,
            ax=0,
            ay=-40,
            bgcolor="rgba(255, 255, 255, 0.8)",
            bordercolor="#c7c7c7",
            borderwidth=1,
            borderpad=4,
            opacity=0.8
        )
    )

# Create buttons for different time periods
button_all = dict(
    label="All Data",
    method="relayout",
    args=[{"xaxis.range": [stock_df.index.min(), stock_df.index.max()]}]
)

button_q1 = dict(
    label="Q1 2022",
    method="relayout",
    args=[{"xaxis.range": ["2022-01-01", "2022-03-31"]}]
)

button_q2 = dict(
    label="Q2 2022",
    method="relayout",
    args=[{"xaxis.range": ["2022-04-01", "2022-06-30"]}]
)

button_q3 = dict(
    label="Q3 2022",
    method="relayout",
    args=[{"xaxis.range": ["2022-07-01", "2022-09-30"]}]
)

button_q4 = dict(
    label="Q4 2022",
    method="relayout",
    args=[{"xaxis.range": ["2022-10-01", "2022-12-31"]}]
)

# Update the layout with buttons and annotations
fig.update_layout(
    title="Interactive Stock Price Analysis with Events",
    xaxis=dict(
        title="Date",
        rangeslider=dict(visible=True),
        type="date"
    ),
    yaxis=dict(
        title="Price ($)"
    ),
    annotations=annotations,
    updatemenus=[
        dict(
            type="buttons",
            direction="right",
            active=0,
            buttons=[button_all, button_q1, button_q2, button_q3, button_q4],
            pad={"r": 10, "t": 10},
            showactive=True,
            x=0.1,
            xanchor="left",
            y=1.1,
            yanchor="top"
        )
    ],
    width=900,
    height=600
)

fig.show()

## Summary and Conclusion

In this notebook, we explored essential techniques for time series visualization:

1. **Data Preparation**: We learned how to handle missing values, resample data to different frequencies, and prepare time series data for effective visualization.

2. **Basic Line Plots**: We created fundamental time series line plots with proper formatting for time-based axes.

3. **Enhanced Visualizations**: We added trend lines, moving averages, annotations, and dual y-axes to provide more context and information in our visualizations.

4. **Date Range Handling**: We demonstrated techniques for visualizing specific time periods and patterns at different frequencies.

5. **Multiple Time Series**: We explored different approaches for comparing multiple time series, including overlays, subplots, and stacked area charts.

6. **Seasonal Decomposition**: We decomposed time series data into trend, seasonal, and residual components to better understand underlying patterns.

7. **Interactive Visualizations**: We created interactive plots with Plotly that allow for exploration, zooming, and enhanced user interaction.

Time series visualization is a crucial skill for data analysis, business intelligence, and forecasting. The techniques covered in this notebook provide a solid foundation for effectively communicating temporal data patterns and insights.

## Further Reading and Resources

- [Pandas Time Series Documentation](https://pandas.pydata.org/pandas-docs/stable/user_guide/timeseries.html)
- [Matplotlib Date Formatting](https://matplotlib.org/stable/gallery/text_labels_and_annotations/date.html)
- [Plotly Time Series Documentation](https://plotly.com/python/time-series/)
- [Seaborn Time Series Visualization](https://seaborn.pydata.org/examples/timeseries_plot.html)
- [Statsmodels Seasonal Decomposition](https://www.statsmodels.org/stable/generated/statsmodels.tsa.seasonal.seasonal_decompose.html)